In [ ]:
!curl https://rclone.org/install.sh | bash

In [ ]:
import os

os.makedirs("/root/.config/rclone", exist_ok=True)

config = """
[wasabi]
type = s3
provider = Wasabi
env_auth = false
access_key_id = 5E9T5EJ27TT7LR3KQHY4
secret_access_key = r2njj2cQsTI1HeSGnIOz1pguY1hz03Me2JCBzTDQ
endpoint = s3.wasabisys.com
"""

with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write(config)

In [ ]:
!rclone copy wasabi:108_VALIS/Old/Pair1/'HE test.ome.tiff' /data/Pair1

In [ ]:
!rclone copy wasabi:108_VALIS/Old/Pair1/'CD8 unmixed IF test.ome.tiff' /data/Pair1

In [ ]:
!rclone copy wasabi:108_VALIS/Old/Pair6 /data/Pair6

In [ ]:
!rclone copy wasabi:108/'In Ammas Lotus Feet'/'Demo Files'/'16-223-35_C15-1.svs' /data 

In [ ]:
#!/usr/bin/env python3

import os
import json
import math
import random
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torchvision.transforms.functional as TF
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

import tifffile
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import torchvision.models as models


# =============================================================================
# Configuration
# =============================================================================
@dataclass
class Config:
    """Configuration for ROSIE-HistoPlexer V6"""
    # Data paths
    data_pairs: List[Tuple[str, str]] = field(default_factory=list)
    pretrained_path: str = ""
    output_dir: str = "output_rosie_histoplexer_v6"
    prediction_save_dir: str = "/kaggle/working/if_predictions"
    
    # Architecture
    input_channels: int = 3
    output_channels: int = -1  # Auto-detect
    fpn_features: int = 256
    
    # Training
    batch_size: int = 8
    num_epochs: int = 200
    patch_size: int = 256
    stride: int = 128
    save_predictions_every: int = 2
    num_visualization_samples: int = 5
    
    # Optimizer
    lr_generator: float = 1e-4  
    lr_discriminator: float = 4e-4  
    beta1: float = 0.5
    beta2: float = 0.999
    weight_decay: float = 1e-4
    
    # Loss weights 
    lambda_pyramid: float = 15.0  
    lambda_patchnce: float = 0.0  
    lambda_adv: float = 5.0 
    lambda_ssim: float = 3.0 
    lambda_focal: float = 8.0 
    lambda_perceptual: float = 2.0  
    
    # Pyramid loss
    pyramid_levels: int = 4
    pyramid_weights: List[float] = field(default_factory=lambda: [1.0, 0.5, 0.25, 0.125])
    
    # PatchNCE - NOW OPERATES ON BACKBONE FEATURES
    nce_layers: List[int] = field(default_factory=lambda: [0, 1, 2, 3])  # backbone stage indices
    nce_temperature: float = 0.07
    num_patches: int = 256
    
    # Training strategy
    freeze_backbone_epochs: int = 5 
    warmup_epochs: int = 5
    
    # Augmentation
    augment: bool = True
    
    # Channel configuration
    channel_names: List[str] = field(default_factory=list)
    channel_weights: List[float] = field(default_factory=list)
    structural_channel_indices: List[int] = field(default_factory=list)  # For SSIM
    
    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 4
    seed: int = 42


def auto_configure_channels(config: Config, if_image: np.ndarray) -> Config:
    """Auto-configure channel settings with structural channel awareness"""
    num_channels = if_image.shape[0]
    
    if config.output_channels == -1 or config.output_channels != num_channels:
        print(f"\nAuto-configuring for {num_channels} IF channels...")
        config.output_channels = num_channels
        
        # Channel names (8th is AF)
        default_names = ["DAPI", "CD8", "C1Q", "TIM3", "PD1", "CD163", "LAG3", "AF"]
        
        if num_channels <= len(default_names):
            config.channel_names = default_names[:num_channels]
        else:
            config.channel_names = default_names + [f"Channel_{i+1}" for i in range(len(default_names), num_channels)]
        
        # Identify structural channels for SSIM
        structural_names = ["DAPI", "AF"]
        config.structural_channel_indices = [
            i for i, name in enumerate(config.channel_names) if name in structural_names
        ]
        
        # Adaptive weights
        weights = []
        for name in config.channel_names:
            if name in structural_names:
                weights.append(1.5)  # Structural emphasis
            else:
                weights.append(4.0)  # Rare marker emphasis
        
        config.channel_weights = weights
        print(f"  Channels: {config.channel_names}")
        print(f"  Weights: {config.channel_weights}")
        print(f"  Structural channels (for SSIM): {[config.channel_names[i] for i in config.structural_channel_indices]}\n")
    
    return config


def load_tiff_image(path: str) -> np.ndarray:
    """Robust TIFF loader"""
    img = tifffile.imread(path)
    img = np.array(img, dtype=np.float32)
    
    if img.ndim == 2:
        img = img[np.newaxis, :, :]
    elif img.ndim == 3:
        if img.shape[0] <= 20 and img.shape[0] < img.shape[1]:
            pass  # [C, H, W]
        elif img.shape[2] <= 20 and img.shape[2] < img.shape[1]:
            img = np.transpose(img, (2, 0, 1))
        elif img.shape[2] == 3:
            img = np.transpose(img, (2, 0, 1))
    elif img.ndim == 4:
        img = img.reshape(-1, img.shape[-2], img.shape[-1])
        
    return img


# =============================================================================
# Models
# =============================================================================
class ROSIEBackbone(nn.Module):
    """ROSIE-style ConvNeXt-Tiny backbone with feature extraction"""
    def __init__(self, pretrained_path: Optional[str] = None):
        super().__init__()
        # Use Tiny (Dim 96) to match checkpoint
        weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        convnext = convnext_tiny(weights=weights)
        
        self.feature_dims = [96, 192, 384, 768]
        
        # Map features to stages
        self.stem = convnext.features[0]
        self.stage1 = convnext.features[1]
        self.stage2 = nn.Sequential(convnext.features[2], convnext.features[3])
        self.stage3 = nn.Sequential(convnext.features[4], convnext.features[5])
        self.stage4 = nn.Sequential(convnext.features[6], convnext.features[7])
        
        if pretrained_path and os.path.exists(pretrained_path):
            self._load_rosie_weights(pretrained_path)
            
    def _load_rosie_weights(self, pretrained_path: str):
        print(f"\n{'='*60}")
        print("Loading ROSIE weights (ConvNeXt-Tiny Mode)...")
        print(f"{'='*60}")
        
        try:
            checkpoint = torch.load(pretrained_path, map_location='cpu', weights_only=False)
            state_dict = checkpoint.get('model_state_dict', checkpoint.get('state_dict', checkpoint))
            
            # Clean prefixes
            clean_ckpt = {}
            for k, v in state_dict.items():
                new_k = k.replace("module.", "")
                clean_ckpt[new_k] = v
                
            current_state = self.state_dict()
            loaded_count = 0
            
            # Map 'features.X' to our attributes
            for key, param in clean_ckpt.items():
                target_key = None
                
                # Stem & Stage 1
                if key.startswith("features.0."):
                    target_key = key.replace("features.0.", "stem.0.")
                elif key.startswith("features.1."):
                    target_key = key.replace("features.1.", "stage1.")
                
                # Stage 2
                elif key.startswith("features.2."):
                    target_key = key.replace("features.2.", "stage2.0.")
                elif key.startswith("features.3."):
                    target_key = key.replace("features.3.", "stage2.1.")
                    
                # Stage 3
                elif key.startswith("features.4."):
                    target_key = key.replace("features.4.", "stage3.0.")
                elif key.startswith("features.5."):
                    target_key = key.replace("features.5.", "stage3.1.")
                
                # Stage 4
                elif key.startswith("features.6."):
                    target_key = key.replace("features.6.", "stage4.0.")
                elif key.startswith("features.7."):
                    target_key = key.replace("features.7.", "stage4.1.")
                
                if target_key and target_key in current_state:
                    if current_state[target_key].shape == param.shape:
                        current_state[target_key] = param
                        loaded_count += 1
            
            self.load_state_dict(current_state, strict=False)
            print(f"Successfully loaded {loaded_count} parameters.")
            
            if loaded_count < 100:
                print("WARNING: Low parameter match. Ensure checkpoint is ConvNeXt-Tiny.")
                
        except Exception as e:
            print(f"Error loading weights: {e}")
        print(f"{'='*60}\n")
    
    def forward(self, x: torch.Tensor, return_features: bool = False) -> List[torch.Tensor]:
        """
        Forward pass with optional feature extraction
        Args:
            x: Input tensor
            return_features: If True, return intermediate features
        Returns:
            List of feature maps from each stage
        """
        features = []
        x = self.stem(x)
        x = self.stage1(x)
        features.append(x)
        x = self.stage2(x)
        features.append(x)
        x = self.stage3(x)
        features.append(x)
        x = self.stage4(x)
        features.append(x)
        return features
    
    def freeze(self):
        """Freeze all backbone parameters"""
        for param in self.parameters():
            param.requires_grad = False
        print("✓ Backbone frozen")
    
    def unfreeze(self):
        """Unfreeze all backbone parameters"""
        for param in self.parameters():
            param.requires_grad = True
        print("✓ Backbone unfrozen")


class FPNDecoder(nn.Module):
    """FPN decoder with multi-scale fusion"""
    def __init__(self, encoder_dims: List[int], fpn_dim: int, output_channels: int):
        super().__init__()
        self.lateral_convs = nn.ModuleList([nn.Conv2d(dim, fpn_dim, 1) for dim in encoder_dims])
        self.fpn_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(fpn_dim, fpn_dim, 3, padding=1),
                nn.GroupNorm(32, fpn_dim),
                nn.GELU()
            )
            for _ in encoder_dims
        ])
        
        self.fusion = nn.Sequential(
            nn.Conv2d(fpn_dim * 4, fpn_dim * 2, 3, padding=1),
            nn.GroupNorm(32, fpn_dim * 2),
            nn.GELU(),
            nn.Conv2d(fpn_dim * 2, fpn_dim, 3, padding=1),
            nn.GroupNorm(32, fpn_dim),
            nn.GELU(),
        )
        
        self.prediction_head = nn.Sequential(
            nn.Conv2d(fpn_dim, 128, 3, padding=1),
            nn.GroupNorm(32, 128),
            nn.GELU(),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, output_channels, 1)
        )
        self.final_upsample = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False)
    
    def forward(self, features: List[torch.Tensor]) -> torch.Tensor:
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]
        
        # Top-down pathway
        for i in range(len(laterals) - 1, 0, -1):
            laterals[i-1] = laterals[i-1] + F.interpolate(
                laterals[i], size=laterals[i-1].shape[2:], mode='bilinear', align_corners=False
            )
        
        # Apply FPN convolutions
        fpn_features = [conv(lat) for conv, lat in zip(self.fpn_convs, laterals)]
        
        # Upsample all to same size and concatenate
        target_size = fpn_features[0].shape[2:]
        upsampled = [fpn_features[0]]
        for feat in fpn_features[1:]:
            upsampled.append(F.interpolate(feat, size=target_size, mode='bilinear', align_corners=False))
        
        fused = torch.cat(upsampled, dim=1)
        fused = self.fusion(fused)
        output = self.prediction_head(fused)
        output = self.final_upsample(output)
        return output


class ROSIEHistoPlexerGenerator(nn.Module):
    """Main generator with ROSIE backbone and FPN decoder"""
    def __init__(self, config: Config):
        super().__init__()
        self.backbone = ROSIEBackbone(config.pretrained_path)
        self.decoder = FPNDecoder(
            encoder_dims=self.backbone.feature_dims,
            fpn_dim=config.fpn_features,
            output_channels=config.output_channels
        )
    
    def forward(self, x: torch.Tensor, return_features: bool = False):
        """
        Forward pass
        Args:
            x: Input HE image
            return_features: If True, also return backbone features for PatchNCE
        Returns:
            output: Predicted IF image
            features: (optional) Backbone features
        """
        features = self.backbone(x, return_features=True)
        output = self.decoder(features)
        output = torch.sigmoid(output)
        
        if return_features:
            return output, features
        return output


class MultiscaleDiscriminator(nn.Module):
    """Multi-scale PatchGAN discriminator"""
    def __init__(self, input_channels: int, ndf: int = 64, n_layers: int = 4, num_scales: int = 3):  # INCREASED layers and scales
        super().__init__()
        self.num_scales = num_scales
        self.discriminators = nn.ModuleList([
            self._make_discriminator(input_channels, ndf, n_layers)
            for _ in range(num_scales)
        ])
        self.downsample = nn.AvgPool2d(3, stride=2, padding=1, count_include_pad=False)
    
    def _make_discriminator(self, input_channels: int, ndf: int, n_layers: int) -> nn.Module:
        layers = []
        layers.append(nn.utils.spectral_norm(nn.Conv2d(input_channels, ndf, 4, stride=2, padding=1)))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        
        nf_mult = 1
        for n in range(1, n_layers):
            nf_mult_prev = nf_mult
            nf_mult = min(2 ** n, 8)
            layers.append(nn.utils.spectral_norm(
                nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, 4, stride=2, padding=1)
            ))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
        
        nf_mult_prev = nf_mult
        nf_mult = min(2 ** n_layers, 8)
        layers.append(nn.utils.spectral_norm(
            nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, 4, stride=1, padding=1)
        ))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        layers.append(nn.Conv2d(ndf * nf_mult, 1, 4, stride=1, padding=1))
        return nn.Sequential(*layers)
    
    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        outputs = []
        for i, discriminator in enumerate(self.discriminators):
            outputs.append(discriminator(x))
            if i < self.num_scales - 1:
                x = self.downsample(x)
        return outputs


# =============================================================================
# Losses
# =============================================================================
class GaussianPyramidLoss(nn.Module):
    """Multi-scale Gaussian pyramid loss"""
    def __init__(self, num_levels: int = 4, weights: Optional[List[float]] = None):
        super().__init__()
        self.num_levels = num_levels
        self.weights = weights or [1.0 / (2 ** i) for i in range(num_levels)]
        
        # Create Gaussian kernel
        kernel_size = 5
        sigma = 1.0
        x = torch.arange(kernel_size).float() - kernel_size // 2
        gauss = torch.exp(-x.pow(2) / (2 * sigma ** 2))
        kernel_1d = gauss / gauss.sum()
        kernel_2d = kernel_1d.unsqueeze(0) * kernel_1d.unsqueeze(1)
        self.register_buffer('gaussian_kernel', kernel_2d.unsqueeze(0).unsqueeze(0))
    
    def _gaussian_blur(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        kernel = self.gaussian_kernel.expand(C, 1, -1, -1)
        return F.conv2d(x, kernel, padding=2, groups=C)
    
    def _build_pyramid(self, x: torch.Tensor) -> List[torch.Tensor]:
        pyramid = [x]
        for _ in range(self.num_levels - 1):
            x = self._gaussian_blur(x)
            x = F.avg_pool2d(x, 2)
            pyramid.append(x)
        return pyramid
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred_pyramid = self._build_pyramid(pred)
        target_pyramid = self._build_pyramid(target)
        total_loss = 0.0
        for p, t, w in zip(pred_pyramid, target_pyramid, self.weights):
            total_loss = total_loss + w * F.l1_loss(p, t)
        return total_loss


class BackbonePatchNCELoss(nn.Module):
    """
    TRUE PatchNCE: Operates on backbone features from HE and predicted IF
    Contrasts features at corresponding layers of the shared backbone
    """
    def __init__(self, nce_layers: List[int], temperature: float = 0.07, num_patches: int = 256):
        super().__init__()
        self.nce_layers = nce_layers
        self.temperature = temperature
        self.num_patches = num_patches
        self.mlp_heads = nn.ModuleDict()
    
    def _create_mlp(self, input_dim: int) -> nn.Module:
        return nn.Sequential(
            nn.utils.spectral_norm(nn.Linear(input_dim, 256)),
            nn.ReLU(),
            nn.utils.spectral_norm(nn.Linear(256, 256))
        )
    
    def _sample_patches_safe(self, feat: torch.Tensor, num_patches: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        SAFE patch sampling: ensures num_patches <= H*W
        """
        B, C, H, W = feat.shape
        feat_flat = feat.permute(0, 2, 3, 1).reshape(B, H * W, C)
        
        # CRITICAL: Ensure we don't sample more patches than available
        actual_num_patches = min(num_patches, H * W)
        
        if actual_num_patches < H * W:
            # Random sampling without replacement
            indices = torch.stack([
                torch.randperm(H * W, device=feat.device)[:actual_num_patches]
                for _ in range(B)
            ])
        else:
            # Use all patches
            indices = torch.arange(H * W, device=feat.device).unsqueeze(0).expand(B, -1)
        
        indices_expanded = indices.unsqueeze(-1).expand(-1, -1, C)
        patches = torch.gather(feat_flat, 1, indices_expanded)
        return patches, indices
    
    def forward(self, feat_he: List[torch.Tensor], feat_if: List[torch.Tensor]) -> torch.Tensor:
        """
        Compute PatchNCE loss between HE and IF backbone features
        Args:
            feat_he: Features from HE image through backbone
            feat_if: Features from predicted IF image through backbone
        """
        total_loss = 0.0
        num_layers = 0
        
        for layer_idx in self.nce_layers:
            if layer_idx >= len(feat_he) or layer_idx >= len(feat_if):
                continue
                
            f_he = feat_he[layer_idx]
            f_if = feat_if[layer_idx]
            
            # Ensure spatial dimensions match
            if f_he.shape[2:] != f_if.shape[2:]:
                f_if = F.interpolate(f_if, size=f_he.shape[2:], mode='bilinear', align_corners=False)
            
            B, C, H, W = f_he.shape
            
            # Create MLP head if needed
            layer_key = f'layer_{layer_idx}'
            if layer_key not in self.mlp_heads:
                self.mlp_heads[layer_key] = self._create_mlp(C).to(f_he.device)
            
            mlp = self.mlp_heads[layer_key]
            
            # SAFE patch sampling
            he_patches, indices = self._sample_patches_safe(f_he, self.num_patches)
            
            # Sample corresponding patches from IF features
            if_flat = f_if.permute(0, 2, 3, 1).reshape(B, H * W, C)
            if_patches = torch.gather(if_flat, 1, indices.unsqueeze(-1).expand(-1, -1, C))
            
            # Project and normalize
            he_proj = F.normalize(mlp(he_patches), dim=-1)
            if_proj = F.normalize(mlp(if_patches), dim=-1)
            
            # Compute similarity matrix
            logits = torch.bmm(he_proj, if_proj.transpose(1, 2)) / self.temperature
            
            # FIXED: Use reshape instead of view for safety
            actual_num_patches = he_patches.shape[1]
            logits = logits.reshape(-1, actual_num_patches)
            labels = torch.arange(actual_num_patches, device=logits.device).unsqueeze(0).expand(B, -1).reshape(-1)
            
            nce_loss = F.cross_entropy(logits, labels)
            total_loss = total_loss + nce_loss
            num_layers += 1
        
        return total_loss / max(num_layers, 1)


class StructuralSSIMLoss(nn.Module):
    """
    SSIM loss computed ONLY on structural channels (DAPI, AF)
    """
    def __init__(self, window_size: int = 11, structural_indices: List[int] = [0, 7]):
        super().__init__()
        self.window_size = window_size
        self.structural_indices = structural_indices
        self.channel = 1
        self.window = self.create_window(window_size, self.channel)

    def gaussian(self, window_size, sigma):
        gauss = torch.Tensor([math.exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
        return gauss/gauss.sum()

    def create_window(self, window_size, channel):
        _1D_window = self.gaussian(window_size, 1.5).unsqueeze(1)
        _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
        window = torch.autograd.Variable(_2D_window.expand(channel, 1, window_size, window_size).contiguous())
        return window

    def _compute_ssim(self, img1, img2):
        """Compute SSIM for single-channel images"""
        (_, channel, _, _) = img1.size()
        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = self.create_window(self.window_size, channel)
            if img1.is_cuda:
                window = window.cuda(img1.get_device())
            window = window.type_as(img1)
            self.window = window
            self.channel = channel

        mu1 = F.conv2d(img1, window, padding=self.window_size//2, groups=channel)
        mu2 = F.conv2d(img2, window, padding=self.window_size//2, groups=channel)
        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1*mu2
        sigma1_sq = F.conv2d(img1*img1, window, padding=self.window_size//2, groups=channel) - mu1_sq
        sigma2_sq = F.conv2d(img2*img2, window, padding=self.window_size//2, groups=channel) - mu2_sq
        sigma12 = F.conv2d(img1*img2, window, padding=self.window_size//2, groups=channel) - mu1_mu2

        C1 = 0.01**2
        C2 = 0.03**2
        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2))/((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
        return 1 - ssim_map.mean()

    def forward(self, pred, target):
        """
        Compute SSIM only on structural channels
        """
        total_loss = 0.0
        for idx in self.structural_indices:
            if idx < pred.shape[1]:
                pred_ch = pred[:, idx:idx+1, :, :]
                target_ch = target[:, idx:idx+1, :, :]
                total_loss = total_loss + self._compute_ssim(pred_ch, target_ch)
        
        return total_loss / len(self.structural_indices)


class FocalLoss(nn.Module):
    """Focal loss for handling class imbalance"""
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        inputs = torch.clamp(inputs, 1e-7, 1-1e-7)
        bce_loss = - (targets * torch.log(inputs) + (1 - targets) * torch.log(1 - inputs))
        pt = torch.exp(-bce_loss)
        f_loss = (1 - pt) ** self.gamma * bce_loss
        return f_loss.mean()


class LSGANLoss(nn.Module):
    """Least-squares GAN loss for stable training"""
    def forward(self, pred, target_is_real):
        target = torch.ones_like(pred) if target_is_real else torch.zeros_like(pred)
        return F.mse_loss(pred, target)


class VGGPerceptualLoss(nn.Module):
    """VGG-based perceptual loss for texture/detail preservation"""
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(pretrained=True).features
        self.blocks = nn.ModuleList([
            vgg[:4],   # relu1_2
            vgg[4:9],  # relu2_2
            vgg[9:18], # relu3_4
            vgg[18:27],# relu4_4
        ])
        for block in self.blocks:
            for param in block.parameters():
                param.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    
    def _preprocess(self, x):
        """Convert single-channel to RGB and normalize for VGG"""
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        x = (x - self.mean) / self.std
        return x
    
    def forward(self, pred, target):
        """Compute perceptual loss on DAPI channel (most structural info)"""
        # Use only DAPI channel (channel 0) for perceptual loss
        pred_dapi = pred[:, 0:1, :, :]
        target_dapi = target[:, 0:1, :, :]
        
        pred_dapi = self._preprocess(pred_dapi)
        target_dapi = self._preprocess(target_dapi)
        
        loss = 0.0
        x_pred = pred_dapi
        x_target = target_dapi
        
        for block in self.blocks:
            x_pred = block(x_pred)
            x_target = block(x_target)
            loss += F.l1_loss(x_pred, x_target)
        
        return loss / len(self.blocks)


class CombinedLoss(nn.Module):
    """Combined loss with all components"""
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.pyramid_loss = GaussianPyramidLoss(
            num_levels=config.pyramid_levels,
            weights=config.pyramid_weights
        )
        # REMOVED: PatchNCE (was causing blur and collapse)
        self.ssim_loss = StructuralSSIMLoss(
            structural_indices=config.structural_channel_indices
        )
        self.focal_loss = FocalLoss(gamma=2.0)
        self.perceptual_loss = VGGPerceptualLoss()  # NEW
        self.gan_loss = LSGANLoss()
        
        # Channel weights for pyramid loss
        self.register_buffer('channel_weights', torch.tensor(config.channel_weights).view(1, -1, 1, 1))
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor,
                feat_he: Optional[List[torch.Tensor]] = None,
                feat_if: Optional[List[torch.Tensor]] = None) -> Dict[str, torch.Tensor]:
        losses = {}
        
        if pred.shape != target.shape:
            target = F.interpolate(target, size=pred.shape[2:], mode='bilinear', align_corners=False)
        
        # Weighted pyramid loss
        weighted_pred = pred * self.channel_weights
        weighted_target = target * self.channel_weights
        losses['pyramid'] = self.pyramid_loss(weighted_pred, weighted_target) * self.config.lambda_pyramid
        
        # REMOVED: PatchNCE
        losses['patchnce'] = torch.tensor(0.0, device=pred.device)
        
        # SSIM on structural channels only
        losses['ssim'] = self.ssim_loss(pred, target) * self.config.lambda_ssim
        
        # Focal loss
        losses['focal'] = self.focal_loss(pred, target) * self.config.lambda_focal
        
        # VGG perceptual loss (NEW)
        losses['perceptual'] = self.perceptual_loss(pred, target) * self.config.lambda_perceptual
        
        losses['total'] = sum(losses.values())
        return losses


# =============================================================================
# Dataset & Visualization
# =============================================================================
class VirtualStainingDataset(Dataset):
    """Dataset for H&E to IF virtual staining"""
    def __init__(self, data_pairs, config, patch_size=256, stride=128, augment=True, if_stats=None):
        self.data_pairs = data_pairs
        self.config = config
        self.patch_size = patch_size
        self.stride = stride
        self.augment = augment
        self.if_stats = if_stats
        self.loaded_images = []
        
        print("\nLoading images...")
        for he_path, if_path in data_pairs:
            he_img = load_tiff_image(he_path)
            if_img = load_tiff_image(if_path)
            
            if len(self.loaded_images) == 0:
                self.config = auto_configure_channels(self.config, if_img)
            
            # Normalize HE
            if he_img.max() > 1.0:
                he_img /= 255.0
            
            self.loaded_images.append((he_img, if_img))
        
        if self.if_stats is None:
            self.if_stats = self._compute_if_stats()
        
        self.patches = self._extract_patches()

    def _compute_if_stats(self):
        print("Computing IF stats (P1/P99)...")
        all_data = []
        for _, if_img in self.loaded_images:
            flat = if_img.reshape(if_img.shape[0], -1)
            idx = np.random.choice(flat.shape[1], min(50000, flat.shape[1]), replace=False)
            all_data.append(flat[:, idx])
        all_data = np.concatenate(all_data, axis=1)
        return {
            'p1': np.percentile(all_data, 1, axis=1).tolist(),
            'p99': np.percentile(all_data, 99, axis=1).tolist()
        }

    def _extract_patches(self):
        patches = []
        for i, (he, _) in enumerate(self.loaded_images):
            _, H, W = he.shape
            pair_patches = []
            for y in range(0, H - self.patch_size + 1, self.stride):
                for x in range(0, W - self.patch_size + 1, self.stride):
                    pair_patches.append({'pair_idx': i, 'y': y, 'x': x})
            
            # Limit to 10000 patches per pair
            if len(pair_patches) > 10000:
                import random
                pair_patches = random.sample(pair_patches, 10000)
                print(f"  Pair {i+1}: Sampled 10000 patches from {len(pair_patches)} available")
            else:
                print(f"  Pair {i+1}: Using all {len(pair_patches)} patches")
            
            patches.extend(pair_patches)
        return patches

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        info = self.patches[idx]
        he, if_img = self.loaded_images[info['pair_idx']]
        y, x = info['y'], info['x']
        
        he_p = he[:, y:y+self.patch_size, x:x+self.patch_size]
        if_p = if_img[:, y:y+self.patch_size, x:x+self.patch_size]
        
        # Normalize IF
        p1 = np.array(self.if_stats['p1']).reshape(-1, 1, 1)
        p99 = np.array(self.if_stats['p99']).reshape(-1, 1, 1)
        if_p = (if_p - p1) / (p99 - p1 + 1e-8)
        if_p = np.clip(if_p, 0, 1)
        
        # Augment
        if self.augment:
            if random.random() > 0.5:
                he_p = np.flip(he_p, axis=2).copy()
                if_p = np.flip(if_p, axis=2).copy()
            if random.random() > 0.5:
                he_p = np.flip(he_p, axis=1).copy()
                if_p = np.flip(if_p, axis=1).copy()
            k = random.randint(0, 3)
            if k > 0:
                he_p = np.rot90(he_p, k, axes=(1,2)).copy()
                if_p = np.rot90(if_p, k, axes=(1,2)).copy()
        
        return {
            'he': torch.from_numpy(he_p).float(),
            'if': torch.from_numpy(if_p).float(),
            'patch_idx': idx  # For tracking samples
        }


def save_visualization(he_batch, pred_batch, gt_batch, epoch, sample_indices, save_dir, channel_names):
    """
    Save multi-sample visualization with side-by-side channel display
    Args:
        he_batch: HE images [B, 3, H, W]
        pred_batch: Predicted IF [B, C, H, W]
        gt_batch: Ground truth IF [B, C, H, W]
        epoch: Current epoch
        sample_indices: List of sample numbers
        save_dir: Output directory
        channel_names: List of channel names
    """
    epoch_dir = save_dir / f"epoch_{epoch}"
    epoch_dir.mkdir(parents=True, exist_ok=True)
    
    num_samples = min(len(sample_indices), he_batch.shape[0])
    num_channels = pred_batch.shape[1]
    
    for i in range(num_samples):
        sample_dir = epoch_dir / f"sample_{sample_indices[i]}"
        sample_dir.mkdir(exist_ok=True)
        
        # Save HE
        he_img = he_batch[i].cpu().numpy().transpose(1, 2, 0)
        he_img = np.clip(he_img, 0, 1)
        plt.imsave(sample_dir / "he.png", he_img)
        
        # Save prediction (side-by-side channels)
        pred_channels = pred_batch[i].cpu().numpy()
        pred_vis = create_channel_visualization(pred_channels, channel_names)
        plt.imsave(sample_dir / "pred.png", pred_vis, cmap='gray')
        
        # Save ground truth (side-by-side channels)
        gt_channels = gt_batch[i].cpu().numpy()
        gt_vis = create_channel_visualization(gt_channels, channel_names)
        plt.imsave(sample_dir / "gt.png", gt_vis, cmap='gray')
    
    print(f"✓ Saved {num_samples} visualization samples to {epoch_dir}")


def create_channel_visualization(channels, channel_names):
    """
    Create side-by-side visualization of all channels
    Args:
        channels: [C, H, W] numpy array
        channel_names: List of channel names
    Returns:
        Tiled image showing all channels
    """
    num_channels = channels.shape[0]
    H, W = channels.shape[1], channels.shape[2]
    
    # Normalize each channel independently
    normalized_channels = []
    for c in range(num_channels):
        ch = channels[c]
        ch_min, ch_max = ch.min(), ch.max()
        if ch_max > ch_min:
            ch_norm = (ch - ch_min) / (ch_max - ch_min)
        else:
            ch_norm = np.zeros_like(ch)
        normalized_channels.append(ch_norm)
    
    # Create tiled image
    tiled = np.concatenate(normalized_channels, axis=1)
    
    return tiled


# =============================================================================
# Trainer
# =============================================================================
class Trainer:
    """Main training class with all improvements"""
    def __init__(self, config: Config):
        self.config = config
        self.device = torch.device(config.device)
        self.output_dir = Path(config.output_dir) / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self.output_dir.mkdir(parents=True, exist_ok=True)
        (self.output_dir / "predictions").mkdir(exist_ok=True)
        
        # Models
        self.generator = ROSIEHistoPlexerGenerator(config).to(self.device)
        self.discriminator = MultiscaleDiscriminator(
            config.input_channels + config.output_channels
        ).to(self.device)
        
        # Loss
        self.criterion = CombinedLoss(config).to(self.device)
        self.gan_loss = LSGANLoss()
        
        # Optimizers
        self.opt_G = AdamW(
            self.generator.parameters(),
            lr=config.lr_generator,
            betas=(config.beta1, config.beta2),
            weight_decay=config.weight_decay
        )
        self.opt_D = AdamW(
            self.discriminator.parameters(),
            lr=config.lr_discriminator,
            betas=(config.beta1, config.beta2),
            weight_decay=config.weight_decay
        )
        
        self.step = 0
        self.current_epoch = 0
        
        # Freeze backbone initially
        if config.freeze_backbone_epochs > 0:
            self.generator.backbone.freeze()
            self.backbone_frozen = True
            print(f"\n{'='*60}")
            print(f"Backbone will be frozen for first {config.freeze_backbone_epochs} epochs")
            print(f"{'='*60}\n")
        else:
            self.backbone_frozen = False

    def train_epoch(self, loader, epoch):
        """Train for one epoch"""
        # Check if we need to unfreeze backbone
        if self.backbone_frozen and epoch >= self.config.freeze_backbone_epochs:
            print(f"\n{'='*60}")
            print(f"Unfreezing backbone at epoch {epoch+1}")
            print(f"{'='*60}\n")
            self.generator.backbone.unfreeze()
            self.backbone_frozen = False
        
        self.generator.train()
        self.discriminator.train()
        stats = defaultdict(float)
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
        
        for batch_idx, batch in enumerate(pbar):
            he = batch['he'].to(self.device)
            target = batch['if'].to(self.device)
            
            # ==================== Discriminator Update ====================
            self.opt_D.zero_grad()
            
            # Generate predictions (detached for discriminator)
            with torch.no_grad():
                pred = self.generator(he, return_features=False)
            
            real_pair = torch.cat([he, target], 1)
            fake_pair = torch.cat([he, pred.detach()], 1)
            
            loss_d = 0
            for out in self.discriminator(real_pair):
                loss_d += self.gan_loss(out, True)
            for out in self.discriminator(fake_pair):
                loss_d += self.gan_loss(out, False)
            loss_d *= 0.5
            
            loss_d.backward()
            self.opt_D.step()
            
            # ==================== Generator Update ====================
            self.opt_G.zero_grad()
            
            # Forward pass (no need for feature extraction anymore)
            pred = self.generator(he, return_features=False)
            
            # Compute all losses (no features needed)
            losses = self.criterion(pred, target)
            
            # Adversarial loss
            loss_adv = 0
            fake_pair = torch.cat([he, pred], 1)
            for out in self.discriminator(fake_pair):
                loss_adv += self.gan_loss(out, True)
            
            total_g = losses['total'] + loss_adv * self.config.lambda_adv
            total_g.backward()
            
            # CRITICAL: Clip gradients to prevent instability
            torch.nn.utils.clip_grad_norm_(self.generator.parameters(), max_norm=1.0)
            
            self.opt_G.step()
            
            # Update statistics
            self.step += 1
            for k, v in losses.items():
                stats[k] += v.item()
            stats['d_loss'] += loss_d.item()
            stats['adv_loss'] += loss_adv.item()
            
            pbar.set_postfix({
                'G': f"{total_g.item():.3f}",
                'D': f"{loss_d.item():.4f}"
            })
        
        return {k: v / len(loader) for k, v in stats.items()}

    def save_predictions(self, loader, epoch):
        """Save visualization samples"""
        self.generator.eval()
        save_dir = self.output_dir / "predictions"
        
        # Collect samples
        all_he = []
        all_pred = []
        all_gt = []
        all_indices = []
        
        with torch.no_grad():
            for batch in loader:
                he = batch['he'].to(self.device)
                gt = batch['if'].to(self.device)
                pred = self.generator(he, return_features=False)
                
                all_he.append(he)
                all_pred.append(pred)
                all_gt.append(gt)
                all_indices.extend(batch['patch_idx'].tolist())
                
                if len(all_indices) >= self.config.num_visualization_samples:
                    break
        
        # Concatenate
        all_he = torch.cat(all_he, dim=0)
        all_pred = torch.cat(all_pred, dim=0)
        all_gt = torch.cat(all_gt, dim=0)
        
        # Select random samples
        num_samples = min(self.config.num_visualization_samples, len(all_indices))
        sample_indices = list(range(num_samples))
        
        save_visualization(
            all_he[:num_samples],
            all_pred[:num_samples],
            all_gt[:num_samples],
            epoch,
            sample_indices,
            save_dir,
            self.config.channel_names
        )

    def save_checkpoint(self, epoch):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'generator_state_dict': self.generator.state_dict(),
            'discriminator_state_dict': self.discriminator.state_dict(),
            'opt_G_state_dict': self.opt_G.state_dict(),
            'opt_D_state_dict': self.opt_D.state_dict(),
            'config': self.config,
        }
        save_path = self.output_dir / f"checkpoint_epoch_{epoch}.pth"
        torch.save(checkpoint, save_path)
        
        # Also save latest
        latest_path = self.output_dir / "latest_model.pth"
        torch.save(self.generator.state_dict(), latest_path)
        
        print(f"✓ Saved checkpoint to {save_path}")


def train(config):
    """Main training function"""
    # Set random seed
    torch.manual_seed(config.seed)
    np.random.seed(config.seed)
    random.seed(config.seed)
    
    # Create dataset and loader
    dataset = VirtualStainingDataset(
        config.data_pairs,
        config,
        patch_size=config.patch_size,
        stride=config.stride,
        augment=config.augment
    )
    
    loader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=True
    )
    
    # Create trainer
    trainer = Trainer(config)
    
    # Save config
    with open(trainer.output_dir / "config.json", 'w') as f:
        json.dump(vars(config), f, indent=2)
    
    print(f"\n{'='*60}")
    print("Starting Training")
    print(f"{'='*60}")
    print(f"Output directory: {trainer.output_dir}")
    print(f"Total epochs: {config.num_epochs}")
    print(f"Batch size: {config.batch_size}")
    print(f"Number of patches: {len(dataset)}")
    print(f"{'='*60}\n")
    
    # Training loop
    for epoch in range(config.num_epochs):
        metrics = trainer.train_epoch(loader, epoch)
        
        # Print metrics
        print(f"\nEpoch {epoch+1}/{config.num_epochs} Metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.6f}")
        
        # Save predictions
        if (epoch + 1) % config.save_predictions_every == 0:
            trainer.save_predictions(loader, epoch + 1)
            trainer.save_checkpoint(epoch + 1)
    
    print(f"\n{'='*60}")
    print("Training Complete!")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    config = Config()
    
    # ================= UPDATE YOUR PATHS HERE =================
    config.data_pairs = [
        ('/data/Pair1/HE test.ome.tiff', '/data/Pair1/CD8 unmixed IF test.ome.tiff'),
    ]
    config.pretrained_path = '/kaggle/working/ROSIE/best_model_single.pth'
    
    # Training configuration
    config.num_epochs = 200
    config.batch_size = 8
    config.save_predictions_every = 2
    config.num_visualization_samples = 5
    config.freeze_backbone_epochs = 3 
    
    # Loss weights (REVISED - No PatchNCE)
    config.lambda_pyramid = 15.0
    config.lambda_patchnce = 0.0  # DISABLED
    config.lambda_adv = 5.0
    config.lambda_ssim = 3.0
    config.lambda_focal = 8.0
    config.lambda_perceptual = 2.0  
    # ==========================================================
    
    if config.data_pairs:
        train(config)
    else:
        print("Error: Please specify data_pairs in the configuration!")

In [ ]:
# INFERENCE 

#     Reference image = "/data/Pair1/HE test.ome.tiff"
#     Inferred on = "/data/16-223-35_C15-1.svs"
#     Patch size : 1024

In [ ]:
#!/usr/bin/env python3
"""
ROSIE-HistoPlexer V6: Inference Script with Stain Normalization
---------------------------------------------------------------
Tests the trained model on patches from validation slide
Applies stain normalization using reference slide as target
Generates visualizations of HE input and Predictions
"""

import os
import random
from pathlib import Path
from typing import List, Tuple
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

import tifffile
import openslide
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from tqdm import tqdm
from skimage import color, exposure, filters, morphology
from scipy import ndimage


# =============================================================================
# Config class
# =============================================================================
@dataclass
class Config:
    """Configuration for ROSIE-HistoPlexer V6 (minimal for inference)"""
    data_pairs: List[Tuple[str, str]] = field(default_factory=list)
    pretrained_path: str = ""
    output_dir: str = "output_rosie_histoplexer_v6"
    prediction_save_dir: str = "/kaggle/working/if_predictions"
    input_channels: int = 3
    output_channels: int = -1
    fpn_features: int = 256
    batch_size: int = 8
    num_epochs: int = 200
    patch_size: int = 256
    stride: int = 128
    save_predictions_every: int = 2
    num_visualization_samples: int = 5
    lr_generator: float = 1e-4
    lr_discriminator: float = 4e-4
    beta1: float = 0.5
    beta2: float = 0.999
    weight_decay: float = 1e-4
    lambda_pyramid: float = 15.0
    lambda_patchnce: float = 0.0
    lambda_adv: float = 5.0
    lambda_ssim: float = 3.0
    lambda_focal: float = 8.0
    lambda_perceptual: float = 2.0
    pyramid_levels: int = 4
    pyramid_weights: List[float] = field(default_factory=lambda: [1.0, 0.5, 0.25, 0.125])
    nce_layers: List[int] = field(default_factory=lambda: [0, 1, 2, 3])
    nce_temperature: float = 0.07
    num_patches: int = 256
    freeze_backbone_epochs: int = 5
    warmup_epochs: int = 5
    augment: bool = True
    channel_names: List[str] = field(default_factory=list)
    channel_weights: List[float] = field(default_factory=list)
    structural_channel_indices: List[int] = field(default_factory=list)
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 4
    seed: int = 42


# =============================================================================
# Stain Normalization
# =============================================================================
class UltimateStainNormalizer:
    """
    Maximum aggression normalization combining all techniques.
    """

    def __init__(self):
        self.stain_matrix_target = None
        self.max_conc_target = None
        self.target_hsv_stats = None
        self.target_rgb_histograms = None

    def _rgb_to_od(self, img):
        img = img.astype(np.float64).clip(1, 255)
        return -np.log(img / 255.0)

    def _od_to_rgb(self, od):
        rgb = 255.0 * np.exp(-od)
        return rgb.clip(0, 255).astype(np.uint8)

    def _get_stain_matrix(self, img, od_threshold=0.15):
        """Extract stain matrix using Macenko method."""
        od = self._rgb_to_od(img)
        od_flat = od.reshape(-1, 3)
        
        od_magnitude = np.sqrt(np.sum(od_flat ** 2, axis=1))
        mask = od_magnitude > od_threshold
        od_filtered = od_flat[mask]

        if od_filtered.shape[0] < 100:
            raise ValueError("Too few tissue pixels")

        cov = np.cov(od_filtered.T)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        V = eigenvectors[:, -2:].T

        for i in range(2):
            if V[i].sum() < 0:
                V[i] = -V[i]

        projections = od_filtered @ V.T
        angles = np.arctan2(projections[:, 1], projections[:, 0])
        
        min_angle = np.percentile(angles, 1)
        max_angle = np.percentile(angles, 99)

        vec1 = np.array([np.cos(min_angle), np.sin(min_angle)]) @ V
        vec2 = np.array([np.cos(max_angle), np.sin(max_angle)]) @ V

        vec1 = vec1 / np.linalg.norm(vec1)
        vec2 = vec2 / np.linalg.norm(vec2)

        # Order by blue/red ratio (H first, E second)
        ratio1 = vec1[2] / (vec1[0] + 1e-10)
        ratio2 = vec2[2] / (vec2[0] + 1e-10)

        if ratio1 > ratio2:
            return np.array([vec1, vec2])
        else:
            return np.array([vec2, vec1])

    def _get_concentrations(self, img, stain_matrix):
        od = self._rgb_to_od(img)
        od_flat = od.reshape(-1, 3)
        return od_flat @ np.linalg.pinv(stain_matrix)

    def fit(self, images):
        """
        Fit on target images with comprehensive statistics.
        """
        print("  Fitting stain normalizer on reference patches...")
        
        # Extract stain matrices
        stain_matrices = []
        max_concs = []
        
        for img in images:
            try:
                sm = self._get_stain_matrix(img)
                conc = self._get_concentrations(img, sm)
                mc = np.percentile(conc, 99, axis=0)
                stain_matrices.append(sm)
                max_concs.append(mc)
            except:
                pass

        stain_matrices = np.array(stain_matrices)
        max_concs = np.array(max_concs)

        # Average stain matrix
        self.stain_matrix_target = np.array([
            np.mean(stain_matrices[:, 0, :], axis=0),
            np.mean(stain_matrices[:, 1, :], axis=0)
        ])
        
        self.stain_matrix_target[0] /= np.linalg.norm(self.stain_matrix_target[0])
        self.stain_matrix_target[1] /= np.linalg.norm(self.stain_matrix_target[1])
        
        self.max_conc_target = np.mean(max_concs, axis=0)

        # Compute HSV statistics
        hsv_stats = []
        for img in images:
            hsv = color.rgb2hsv(img)
            hsv_stats.append({
                'h_mean': np.mean(hsv[:, :, 0]),
                'h_std': np.std(hsv[:, :, 0]),
                's_mean': np.mean(hsv[:, :, 1]),
                's_std': np.std(hsv[:, :, 1]),
                'v_mean': np.mean(hsv[:, :, 2]),
                'v_std': np.std(hsv[:, :, 2]),
            })

        # Average HSV stats
        self.target_hsv_stats = {
            'h_mean': np.mean([s['h_mean'] for s in hsv_stats]),
            'h_std': np.mean([s['h_std'] for s in hsv_stats]),
            's_mean': np.mean([s['s_mean'] for s in hsv_stats]),
            's_std': np.mean([s['s_std'] for s in hsv_stats]),
            'v_mean': np.mean([s['v_mean'] for s in hsv_stats]),
            'v_std': np.mean([s['v_std'] for s in hsv_stats]),
        }

        # Compute target RGB histograms for final matching
        all_pixels = np.concatenate([img.reshape(-1, 3) for img in images[:10]], axis=0)
        self.target_rgb_histograms = []
        for ch in range(3):
            hist, _ = np.histogram(all_pixels[:, ch], bins=256, range=(0, 255))
            self.target_rgb_histograms.append(hist)

        print(f"    ✓ Fitted on {len(images)} patches")

    def transform(self, img):
        """
        Multi-stage aggressive transformation.
        """
        h, w, c = img.shape
        result = img.copy()

        # STAGE 1: Macenko stain normalization
        try:
            sm_source = self._get_stain_matrix(result)
            conc = self._get_concentrations(result, sm_source)
            mc_source = np.percentile(conc, 99, axis=0)
            
            conc *= (self.max_conc_target / (mc_source + 1e-10))
            od_norm = conc @ self.stain_matrix_target
            result = self._od_to_rgb(od_norm.reshape(h, w, c))
        except:
            pass  # Skip if normalization fails

        # STAGE 2: HSV-based intensity and saturation matching
        hsv = color.rgb2hsv(result)

        # Match saturation
        s_current_mean = np.mean(hsv[:, :, 1])
        s_current_std = np.std(hsv[:, :, 1])
        
        if s_current_std > 0:
            hsv[:, :, 1] = (hsv[:, :, 1] - s_current_mean) / s_current_std
            hsv[:, :, 1] = hsv[:, :, 1] * self.target_hsv_stats['s_std'] + self.target_hsv_stats['s_mean']
            hsv[:, :, 1] = hsv[:, :, 1] * 1.2  # Saturation boost

        hsv[:, :, 1] = np.clip(hsv[:, :, 1], 0, 1)

        # Match value (intensity)
        v_current_mean = np.mean(hsv[:, :, 2])
        v_current_std = np.std(hsv[:, :, 2])
        
        if v_current_std > 0:
            hsv[:, :, 2] = (hsv[:, :, 2] - v_current_mean) / v_current_std
            hsv[:, :, 2] = hsv[:, :, 2] * self.target_hsv_stats['v_std'] + self.target_hsv_stats['v_mean']

        hsv[:, :, 2] = np.clip(hsv[:, :, 2], 0, 1)

        # Convert back to RGB
        result = color.hsv2rgb(hsv)
        result = (result * 255).clip(0, 255).astype(np.uint8)

        # STAGE 3: Contrast matching via adaptive histogram equalization
        try:
            # Apply CLAHE
            lab = color.rgb2lab(result)
            lab[:, :, 0] = exposure.equalize_adapthist(lab[:, :, 0] / 100.0, clip_limit=0.01) * 100.0
            result = color.lab2rgb(lab)
            result = (result * 255).clip(0, 255).astype(np.uint8)
        except:
            pass

        # STAGE 4: Per-channel intensity rescaling
        for ch in range(3):
            p1 = np.percentile(result[:, :, ch], 1)
            p99 = np.percentile(result[:, :, ch], 99)
            
            target_p1 = 20
            target_p99 = 240
            
            if p99 > p1:
                result[:, :, ch] = (result[:, :, ch] - p1) / (p99 - p1) * (target_p99 - target_p1) + target_p1
                result[:, :, ch] = np.clip(result[:, :, ch], 0, 255)

        # STAGE 5: Final histogram matching
        if self.target_rgb_histograms is not None:
            try:
                for ch in range(3):
                    source_hist, _ = np.histogram(result[:, :, ch].flatten(), bins=256, range=(0, 255))
                    target_hist = self.target_rgb_histograms[ch]
                    
                    source_cdf = np.cumsum(source_hist).astype(np.float64)
                    source_cdf /= source_cdf[-1]
                    
                    target_cdf = np.cumsum(target_hist).astype(np.float64)
                    target_cdf /= target_cdf[-1]
                    
                    lut = np.interp(source_cdf, target_cdf, np.arange(256))
                    result[:, :, ch] = lut[result[:, :, ch]]
            except:
                pass

        return result.astype(np.uint8)


# =============================================================================
# Model Architecture
# =============================================================================
class ROSIEBackbone(nn.Module):
    """ROSIE-style ConvNeXt-Tiny backbone"""
    def __init__(self):
        super().__init__()
        weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        convnext = convnext_tiny(weights=weights)
        
        self.feature_dims = [96, 192, 384, 768]
        
        self.stem = convnext.features[0]
        self.stage1 = convnext.features[1]
        self.stage2 = nn.Sequential(convnext.features[2], convnext.features[3])
        self.stage3 = nn.Sequential(convnext.features[4], convnext.features[5])
        self.stage4 = nn.Sequential(convnext.features[6], convnext.features[7])
    
    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        features = []
        x = self.stem(x)
        x = self.stage1(x)
        features.append(x)
        x = self.stage2(x)
        features.append(x)
        x = self.stage3(x)
        features.append(x)
        x = self.stage4(x)
        features.append(x)
        return features


class FPNDecoder(nn.Module):
    """FPN decoder with multi-scale fusion"""
    def __init__(self, encoder_dims: List[int], fpn_dim: int, output_channels: int):
        super().__init__()
        self.lateral_convs = nn.ModuleList([nn.Conv2d(dim, fpn_dim, 1) for dim in encoder_dims])
        self.fpn_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(fpn_dim, fpn_dim, 3, padding=1),
                nn.GroupNorm(32, fpn_dim),
                nn.GELU()
            )
            for _ in encoder_dims
        ])
        
        self.fusion = nn.Sequential(
            nn.Conv2d(fpn_dim * 4, fpn_dim * 2, 3, padding=1),
            nn.GroupNorm(32, fpn_dim * 2),
            nn.GELU(),
            nn.Conv2d(fpn_dim * 2, fpn_dim, 3, padding=1),
            nn.GroupNorm(32, fpn_dim),
            nn.GELU(),
        )
        
        self.prediction_head = nn.Sequential(
            nn.Conv2d(fpn_dim, 128, 3, padding=1),
            nn.GroupNorm(32, 128),
            nn.GELU(),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, output_channels, 1)
        )
        self.final_upsample = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False)
    
    def forward(self, features: List[torch.Tensor]) -> torch.Tensor:
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]
        
        for i in range(len(laterals) - 1, 0, -1):
            laterals[i-1] = laterals[i-1] + F.interpolate(
                laterals[i], size=laterals[i-1].shape[2:], mode='bilinear', align_corners=False
            )
        
        fpn_features = [conv(lat) for conv, lat in zip(self.fpn_convs, laterals)]
        
        target_size = fpn_features[0].shape[2:]
        upsampled = [fpn_features[0]]
        for feat in fpn_features[1:]:
            upsampled.append(F.interpolate(feat, size=target_size, mode='bilinear', align_corners=False))
        
        fused = torch.cat(upsampled, dim=1)
        fused = self.fusion(fused)
        output = self.prediction_head(fused)
        output = self.final_upsample(output)
        return output


class ROSIEHistoPlexerGenerator(nn.Module):
    """Main generator"""
    def __init__(self, output_channels: int, fpn_features: int = 256):
        super().__init__()
        self.backbone = ROSIEBackbone()
        self.decoder = FPNDecoder(
            encoder_dims=self.backbone.feature_dims,
            fpn_dim=fpn_features,
            output_channels=output_channels
        )
    
    def forward(self, x: torch.Tensor):
        features = self.backbone(x)
        output = self.decoder(features)
        output = torch.sigmoid(output)
        return output


# =============================================================================
# Utilities
# =============================================================================
def load_tiff_image(path: str) -> np.ndarray:
    """Load TIFF image"""
    img = tifffile.imread(path)
    img = np.array(img, dtype=np.float32)
    
    if img.ndim == 2:
        img = img[np.newaxis, :, :]
    elif img.ndim == 3:
        if img.shape[0] <= 20 and img.shape[0] < img.shape[1]:
            pass
        elif img.shape[2] <= 20 and img.shape[2] < img.shape[1]:
            img = np.transpose(img, (2, 0, 1))
        elif img.shape[2] == 3:
            img = np.transpose(img, (2, 0, 1))
    elif img.ndim == 4:
        img = img.reshape(-1, img.shape[-2], img.shape[-1])
    
    return img


def create_tissue_mask(thumbnail):
    """Create tissue mask from thumbnail"""
    gray = color.rgb2gray(thumbnail)
    threshold = filters.threshold_otsu(gray)
    mask = gray < threshold
    mask = morphology.remove_small_objects(mask, min_size=500)
    return morphology.remove_small_holes(mask, area_threshold=500)


def extract_random_patches_from_slide(slide, patch_size: Tuple[int, int], 
                                     num_patches: int = 20, level: int = 1,
                                     min_tissue_fraction: float = 0.7):
    """
    Extract random patches from OpenSlide object
    Args:
        slide: OpenSlide object
        patch_size: (height, width) of patches
        num_patches: Number of random patches to extract
        level: Pyramid level to read from
        min_tissue_fraction: Minimum tissue content
    Returns:
        List of (patch, y, x) tuples
    """
    # Get thumbnail for tissue detection
    w, h = slide.dimensions
    target_size = 2000
    ratio = target_size / max(h, w)
    thumb_size = (int(w * ratio), int(h * ratio))
    thumbnail = np.array(slide.get_thumbnail(thumb_size))[:, :, :3]
    
    # Create tissue mask
    tissue_mask = create_tissue_mask(thumbnail)
    
    # Calculate scaling factors
    val_scale = slide.dimensions[0] / thumbnail.shape[1]
    patch_size_thumb = max(int(patch_size[0] / (slide.level_dimensions[level][0] / thumbnail.shape[1])), 10)
    
    patches = []
    attempts = 0
    max_attempts = num_patches * 100
    
    rng = np.random.RandomState(123)
    
    while len(patches) < num_patches and attempts < max_attempts:
        # Random location in thumbnail
        ty = rng.randint(0, max(1, tissue_mask.shape[0] - patch_size_thumb))
        tx = rng.randint(0, max(1, tissue_mask.shape[1] - patch_size_thumb))
        
        # Check tissue content
        if tissue_mask[ty:ty+patch_size_thumb, tx:tx+patch_size_thumb].mean() >= min_tissue_fraction:
            # Convert to full resolution coordinates
            fy = int(ty * val_scale)
            fx = int(tx * val_scale)
            
            try:
                # Read patch
                region = slide.read_region((fx, fy), level, patch_size)
                patch = np.array(region)[:, :, :3]
                
                if patch.shape[0] >= patch_size[0] // 2 and patch.shape[1] >= patch_size[1] // 2:
                    patches.append((patch, fy, fx))
            except Exception as e:
                print(f"  Warning reading patch at ({fx}, {fy}): {e}")
        
        attempts += 1
    
    return patches


def extract_reference_patches(ref_path: str, num_patches: int = 30, 
                              patch_size: int = 256) -> List[np.ndarray]:
    """Extract patches from reference TIFF for normalizer fitting"""
    print(f"\nExtracting reference patches from {os.path.basename(ref_path)}...")
    
    ref_tif = tifffile.TiffFile(ref_path)
    
    # Get thumbnail
    levels = ref_tif.series[0].levels
    for level in levels:
        h, w = level.shape[:2]
        if max(h, w) <= 4000:
            data = level.asarray()[:, :, :3]
            from PIL import Image as PILImage
            pil_img = PILImage.fromarray(data)
            ratio = 2000 / max(h, w)
            new_size = (int(w * ratio), int(h * ratio))
            thumbnail = np.array(pil_img.resize(new_size, PILImage.LANCZOS))
            break
    
    # Create tissue mask
    tissue_mask = create_tissue_mask(thumbnail)
    
    # Calculate scaling
    ref_scale = ref_tif.series[0].levels[0].shape[1] / thumbnail.shape[1]
    patch_size_thumb = max(int(patch_size / ref_scale), 10)
    
    # Extract coordinates
    rng = np.random.RandomState(42)
    coords = []
    attempts = 0
    
    while len(coords) < num_patches and attempts < num_patches * 100:
        ty = rng.randint(0, max(1, tissue_mask.shape[0] - patch_size_thumb))
        tx = rng.randint(0, max(1, tissue_mask.shape[1] - patch_size_thumb))
        
        if tissue_mask[ty:ty+patch_size_thumb, tx:tx+patch_size_thumb].mean() >= 0.7:
            coords.append((tx, ty))
        
        attempts += 1
    
    # Load full resolution data
    full_data = ref_tif.series[0].levels[0].asarray()[:, :, :3]
    
    # Extract patches
    patches = []
    for tx, ty in coords:
        fx = int(tx * ref_scale)
        fy = int(ty * ref_scale)
        
        patch = full_data[fy:fy+patch_size, fx:fx+patch_size].copy()
        if patch.shape[0] >= patch_size // 2 and patch.shape[1] >= patch_size // 2:
            patches.append(patch)
    
    ref_tif.close()
    print(f"  ✓ Extracted {len(patches)} reference patches")
    
    return patches


def create_visualization(he_patch_orig: np.ndarray, he_patch_norm: np.ndarray,
                        pred_patch: np.ndarray, channel_names: List[str], 
                        patch_idx: int, location: Tuple[int, int], save_path: Path):
    """
    Create composite visualization with HE Original, HE Normalized, and all IF channels in a grid
    """
    num_channels = len(channel_names)
    
    # Calculate grid dimensions: 2 columns for HE images + channels in 4 columns
    n_cols = 4
    n_rows = int(np.ceil((num_channels + 2) / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    # Plot HE Original
    he_orig_display = he_patch_orig / 255.0 if he_patch_orig.max() > 1 else he_patch_orig
    he_orig_display = np.clip(he_orig_display, 0, 1)
    axes[0].imshow(he_orig_display)
    axes[0].set_title('HE', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Plot HE Normalized
    he_norm_display = he_patch_norm / 255.0 if he_patch_norm.max() > 1 else he_patch_norm
    he_norm_display = np.clip(he_norm_display, 0, 1)
    axes[1].imshow(he_norm_display)
    axes[1].set_title('HE Normalized', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    # Plot each IF channel
    for ch_idx, ch_name in enumerate(channel_names):
        ax_idx = ch_idx + 2
        pred_ch = pred_patch[ch_idx]
        pred_normalized = (pred_ch - pred_ch.min()) / (pred_ch.max() - pred_ch.min() + 1e-8)
        axes[ax_idx].imshow(pred_normalized, cmap='gray', vmin=0, vmax=1)
        axes[ax_idx].set_title(ch_name, fontsize=12, fontweight='bold')
        axes[ax_idx].axis('off')
    
    # Hide any unused subplots
    for idx in range(num_channels + 2, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  ✓ Saved: {save_path.name}")


# =============================================================================
# Main Inference
# =============================================================================
def run_inference(checkpoint_path: str, reference_path: str, validation_path: str, 
                 output_dir: str, num_patches: int = 20, 
                 patch_size: Tuple[int, int] = (1024, 1024),
                 num_channels: int = 8):
    """
    Run inference on validation slide with stain normalization
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*60}")
    print("ROSIE-HistoPlexer V6 - Inference with Normalization")
    print(f"{'='*60}")
    print(f"Device: {device}")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Reference: {os.path.basename(reference_path)}")
    print(f"Validation: {os.path.basename(validation_path)}")
    print(f"Patch size: {patch_size}")
    print(f"Number of patches: {num_patches}")
    print(f"{'='*60}\n")
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Define channel names
    channel_names = ["DAPI", "CD8", "C1Q", "TIM3", "PD1", "CD163", "LAG3", "AF"][:num_channels]
    
    # Step 1: Extract reference patches and fit normalizer
    ref_patches = extract_reference_patches(reference_path, num_patches=30, patch_size=patch_size[0])
    
    normalizer = UltimateStainNormalizer()
    normalizer.fit(ref_patches)
    
    # Step 2: Load model
    print(f"\nLoading model from checkpoint...")
    model = ROSIEHistoPlexerGenerator(
        output_channels=num_channels,
        fpn_features=256
    ).to(device)
    
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    if 'generator_state_dict' in checkpoint:
        state_dict = checkpoint['generator_state_dict']
    elif 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    else:
        state_dict = checkpoint
    
    model.load_state_dict(state_dict)
    model.eval()
    print("  ✓ Model loaded successfully")
    
    # Step 3: Extract validation patches
    print(f"\nExtracting {num_patches} patches from validation slide...")
    val_slide = openslide.OpenSlide(validation_path)
    val_level = 1 if val_slide.level_count > 1 else 0
    
    patches = extract_random_patches_from_slide(
        val_slide, 
        patch_size=patch_size, 
        num_patches=num_patches,
        level=val_level
    )
    print(f"  ✓ Extracted {len(patches)} patches")
    
    # Step 4: Run inference with normalization
    print(f"\nRunning inference with stain normalization...")
    with torch.no_grad():
        for idx, (he_patch_orig, y, x) in enumerate(tqdm(patches, desc="Processing")):
            # Normalize the patch
            he_patch_norm = normalizer.transform(he_patch_orig)
            
            # Prepare for model (normalize to 0-1)
            he_tensor = torch.from_numpy(he_patch_norm.transpose(2, 0, 1)).unsqueeze(0).float().to(device)
            he_tensor = he_tensor / 255.0  # Normalize to 0-1
            
            # Generate prediction
            pred_tensor = model(he_tensor)
            pred_patch = pred_tensor.squeeze(0).cpu().numpy()
            
            # Create visualization
            save_path = output_path / f"patch_{idx:03d}_y{y}_x{x}.png"
            create_visualization(
                he_patch_orig=he_patch_orig,
                he_patch_norm=he_patch_norm,
                pred_patch=pred_patch,
                channel_names=channel_names,
                patch_idx=idx,
                location=(y, x),
                save_path=save_path
            )
    
    val_slide.close()
    
    print(f"\n{'='*60}")
    print(f"Inference Complete!")
    print(f"Results saved to: {output_path}")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    # Configuration
    checkpoint_path = "/kaggle/working/run_20260131_103946/checkpoint_epoch_56.pth"
    reference_path = "/data/Pair1/HE test.ome.tiff"
    validation_path = "/data/16-223-35_C15-1.svs"
    output_dir = "/kaggle/working/inference_results_normalized_new"
    
    # Run inference
    run_inference(
        checkpoint_path=checkpoint_path,
        reference_path=reference_path,
        validation_path=validation_path,
        output_dir=output_dir,
        num_patches=20,
        patch_size=(1024, 1024),
        num_channels=8
    )